# R2AI Kaggle Inference Notebook
Notebook này được thiết kế để chạy pipeline RAG offline trên Kaggle sử dụng SQLite DB và Local FAISS Index.

In [ ]:
!pip install -q transformers sentence-transformers faiss-gpu underthesea accelerate bitsandbytes

In [ ]:
import os
import sys
import json
import time
import shutil

# 1. Khai báo các đường dẫn trên Kaggle
# ĐIỀU CHỈNH: Tên dataset của bạn trên Kaggle (ví dụ: r2ai-project)
KAGGLE_DATASET_PATH = "/kaggle/input/r2ai-project"

# Thư mục làm việc trên Kaggle
WORKING_DIR = "/kaggle/working"
PROJECT_ROOT = os.path.join(WORKING_DIR, "R2AI")

# 2. Copy toàn bộ code và database từ thư mục Input sang thư mục Working (để có thể ghi file)
if not os.path.exists(PROJECT_ROOT):
    print("Copying project from input to working directory...")
    shutil.copytree(KAGGLE_DATASET_PATH, PROJECT_ROOT)
    print("Copy completed!")

# 3. Thêm project root vào thư mục path để import
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root added to path: {PROJECT_ROOT}")

In [ ]:
# 4. Cấu hình biến môi trường để chạy Offline hoàn toàn
os.environ["HF_HUB_OFFLINE"] = "0"  # Tắt Offline mode để HuggingFace tải weights nếu chưa có trên Kaggle

# Tắt cảnh báo Tokenizer
import logging
import transformers
transformers.logging.set_verbosity_error()
logging.getLogger("transformers.tokenization_utils_base").setLevel(logging.ERROR)

In [ ]:
from src.main_pipeline import LegalRAGPipeline

print("Initializing Pipeline...")
# Sử dụng cấu hình LLM phù hợp với Kaggle. Kaggle P100 có 16GB VRAM.
# Ở đây dùng Qwen2.5-1.5B hoặc các model nhẹ để tránh OOM
pipeline = LegalRAGPipeline(
    use_llm_rewrite=False,
    llm_model_name="Qwen/Qwen2.5-1.5B-Instruct"
)
print("Pipeline initialized!")

In [ ]:
# 5. Đọc file câu hỏi
data_path = os.path.join(PROJECT_ROOT, "R2AIStage1DATA.json")
with open(data_path, "r", encoding="utf-8") as f:
    test_questions = json.load(f)

print(f"Loaded {len(test_questions)} questions.")

results = []
start_time = time.time()

# Có thể giới hạn số câu hỏi để test: test_questions[:5]
for idx, q in enumerate(test_questions):
    qid = q.get("id", idx)
    question = q.get("question", "")

    print(f"\n[{idx+1}/{len(test_questions)}] Processing: {question}")
    t0 = time.time()

    try:
        out = pipeline.run(question)
        
        relevant_docs = []
        docs_seen = set()
        for r in out.get("top5_results", []):
            doc_str = r.format_relevant_doc()
            if doc_str not in docs_seen:
                docs_seen.add(doc_str)
                relevant_docs.append(doc_str)

        relevant_articles = []
        articles_seen = set()
        for r in out.get("top5_results", []):
            if r.article_hint:
                art_str = r.format_relevant_article()
                if art_str not in articles_seen:
                    articles_seen.add(art_str)
                    relevant_articles.append(art_str)

        results.append({
            "id": qid,
            "question": question,
            "answer": out.get("final_answer", ""),
            "relevant_docs": relevant_docs,
            "relevant_articles": relevant_articles
        })
        print(f"Done in {time.time() - t0:.2f}s")
        
    except Exception as e:
        print(f"ERROR processing ID {qid}: {e}")
        results.append({
            "id": qid,
            "question": question,
            "answer": "",
            "relevant_docs": [],
            "relevant_articles": []
        })

total_time = time.time() - start_time
print(f"\nCompleted all questions in {total_time:.2f}s.")

In [ ]:
# 6. Lưu file kết quả xuống thư mục Working
output_path = "/kaggle/working/submission.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Results saved to {output_path}")
print("Bạn có thể download file submission.json từ cột bên phải của Kaggle!")